# Assignment Overview

Links to the notes discussed in the video
* [Model Selection Overview](./ModelSelect.pdf)
* [Model Types](./ModelType.pdf)
* [Model Decision Factors](./ModelDecisionFactors.pdf)
* [Generalization Techiques](./Generalization.pdf)

The assignment consists of two parts requiring you to select appropriate models with associated code/text.

1. Determine challenge and relevant model for two distinct situations (fill out this notebook). 
1. Address the data code needed and the model for [car factors](./CarFactors/carfactors.ipynb) contained in the subdirectory, CarFactors.

* ***Check the rubric in Canvas*** to make sure you understand the requirements and the assocated weights grading

# Part 1: Speed Dating Model Selection

You are to explore the data set on speed dating and construct two models that provide some insight such as grouping or predictions.  The models must come from different model areas such as listed as categories in the [ModelTypes](./ModelTypes.pdf) document.  You must justify your answer considering the data and the prediction value.

The data is contained in [SpeedDatingData.csv](SpeedDatingData.csv).  The values are detailed in [SpeedDatingKey.md](./SpeedDatingKey.md).  The directory also contains the original key document - SpeedDatingDataKey.docx but jupyter lab is unable to render it.  You are free to render it outside of jupyter lab if something didn't translater clearly.  The open source tool [pandoc](https://pandoc.org/installing.html) was used to perform the translation.  It is useful for almost any translation and works in all major operating systems

# Model 1

## Outline the challenge 

How can we predict whether or not a partner would like to be seen again by a participant?  This would be a supervised classification problem using *dec* as the label.

### Select the features and their justification 

We will select all features except those listed below. This is because the desired features contain information up to and capturing the moment of the speed date. The undesired features below will be excluded based on the listed justification:

Since we are predicting, we will exclude all data collected after the start of the dating wave.  This includes all features starting at *attr1_s* and beyond, as well as *match*.
  
We will exclude the following features since they are unique to each speed date:  
*iid, id, idg, condtn, wave, round, position, positin1, partner, pid*  
  
We will exclude the following features since they include too many empty entries:  
*undergra, mn_sat, tuition, zipcode, income, attr4_1* to *shar4_1, attr5_1* to *shar5_1*

We will exclude *field* and *career* since we will later one-hot-encode them.

Note that we are keeping *order* since it may be true that the dating order impacts the participants selection/rating behavior.

### Note necessary feature processing such as getting rid of empty cells etc.

The blank rows in *career_c* can be imputed based on the text in *career*.  

The following will be one-hot-encoded since they are not ordinal:  
*race_o, field_cd, race, goal, career_c*

The following will be re-coded to 0 or 1:  
*met_o, met*

The following will be normalized, since they were scored differently depending on wave (simply divide by sum score):  
*attr1_1 to shar1_1, attr2_1 to shar2_1*  

Any remaining empty cells will result in a deleted row.


### Model Selection

We will use a random forest, which uses decision trees in an ensemble.  
A decision tree was selected as the ensemble theme because decision trees count occurences and place observations into bins.  
Ensembled into a random forest, the model will be resilient against variance and perform quite well.

In [ ]:
# Enter python code of constructing your selected model  - CODE REQUIRED! (only the model creation)

# We will start assuming the feature and label data have already been processed
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)  # generate train/test split
max_depth = 10  # we would adjust this hyperparameter to tune the model
clf = RandomForestClassifier(max_depth=max_depth, n_jobs=-1)  # initiate the model
clf.fit(X_train, y_train)  # train the model

# Model 2

## Outline the challenge

How can we quantify the extent in which a partner is liked by a participant?  This would be a supervised regression problem using *like* as the label.

### Select the features and their justification

The selection mechanism is the same as in model 1, because in both challenges we are making a prediction for the outcome of a speed date.  

We will select all features except those listed below. This is because the desired features contain information up to and capturing the moment of the speed date. The undesired features below will be excluded based on the listed justification:

Since we are predicting, we will exclude all data collected after the start of the dating wave.  This includes all features starting at *attr1_s* and beyond, as well as *match*.
  
We will exclude the following features since they are unique to each speed date:  
*iid, id, idg, condtn, wave, round, position, positin1, partner, pid*  
  
We will exclude the following features since they include too many empty entries:  
*undergra, mn_sat, tuition, zipcode, income, attr4_1* to *shar4_1, attr5_1* to *shar5_1*

We will exclude *field* and *career* since we will later one-hot-encode them.

Note that we are keeping *order* since it may be true that the dating order impacts the participants selection/rating behavior.

### Note necessary feature processing such as getting rid of empty cells etc.

The feature processing is nearly the same as in model 1, because in both challenges we are making a prediction for the outcome of a speed date, and in both challenges we must use a model that relies on numerical input (whether ordinal or not).  The difference between this and model 1 will be later described. 

The blank rows in *career_c* can be imputed based on the text in *career*.  

The following will be one-hot-encoded since they are not ordinal:  
*race_o, field_cd, race, goal, career_c*

The following will be re-coded to 0 or 1:  
*met_o, met*

The following will be normalized, since they were scored differently depending on wave (simply divide by sum score):  
*attr1_1 to shar1_1, attr2_1 to shar2_1*  

Any remaining empty cells will result in a deleted row.

The difference between this and model 1 is that we will further normalize the features here (scale all non-binary values as float from 0 to 1), which is optimal for an artificial neural network.  

### Model Selection

Outline the rationale for selecting the model noting how its capabilities address your challenge

We will use a fully connected artifical neural network (FCN/ANN).  
An ANN will learn the minimum cost order of regression and weights.  
Such flexibility should best capture the true relationship between features and the label (whereas other regression models assume a certain order of regression - linear/polynomial, etc).  
We will not normalize the label *like*.

In [1]:
# Enter python code of constructing your selected model  - CODE REQUIRED! (only the model creation)

# We will start assuming the feature and label data have already been processed
import tensorflow.keras as tfk
import tensorflow.keras.layers as tfl
from sklearn.model_selection import RandomizedSearchCV, train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)  # generate train/test split

def simple_model(input_shape):  # define the FCN/ANN
    """
    A simple fully connected neural network
    
    We select 3x layers because 2x layers can approximate any function,
    and we would like an additional level of learning while minimally
    risking over-learning and a vanishing gradient.
    
    We implement dropout to reduce over-learning and increase
    model robustness.
    
    Arguments:
    input_shape -- the shape of a single input

    Returns:
    model -- TF Keras model
    """
    x = tfk.Input(shape=input_shape)
    x = tfl.Dense(units=128, activation='relu')(x)
    x = tfl.Dropout(0.5)
    x = tfl.Dense(units=64, activation='relu')(x)
    x = tfl.Dropout(0.5)
    x = tfl.Dense(units=32, activation='relu')(x)
    x = tfl.Dropout(0.5)
    outputs = tfl.Dense(units=1)(x)
    model = tf.keras.Model(inputs=input_dat, outputs=outputs)
    return model

model = simple_model()  # initialize model

# assign loss and metrics appropriate to nonlinear regression
happy_model.compile(optimizer='adam',
                   loss='mean_squared_error',
                   metrics=['mean_squared_error'])

happy_model.fit(X_train, Y_train, epochs=100, batch_size=16)  # train the model